# SeizeIT2 window/horizon sweep — Colab training

Trains the 9 `BaselineEEGNet` window/horizon combinations produced by
`scripts/seizeit2/run_window_horizon_sweep.py --preprocess-only --package`,
using the already-preprocessed data packaged under `outputs/colab_packages/`.

**This notebook only trains — it does not preprocess.** Preprocessing (filtering,
labeling, standardization) already happened on the machine that produced the
`colab_packages` zips; see the "Preprocessing on one machine, training on
another" section of `run_window_horizon_sweep.py` for why that split exists.

**Before running this notebook:**
1. Open it with the Colab VSCode extension and connect to a GPU runtime
   (`Runtime type > T4 GPU` or better).
2. Upload the whole `outputs/colab_packages/<sweep-name>/` folder from your
   local machine to Google Drive. For this sweep that folder holds nine small
   `w{window}_h{horizon}.zip` files (~28 MB each),
   `shared_standardized_recordings.zip` (78.7 GB), and
   `shared_standardized_recordings_supplement.zip` (0.29 GB).
3. Point `DRIVE_PACKAGES_DIR` in the **Configuration** cell at that folder.

Then run top to bottom. Step 5a checks the upload in seconds, so a wrong path
or a truncated file surfaces immediately rather than an hour into extraction.

**Why one notebook trains all nine combos.** Splitting the sweep across nine
notebooks looks like it would cut disk usage, but it would not: the combos
differ only in their decision labels, and all of them read from the same pool of
standardized recordings. That is why each combo zip is ~28 MB while the shared
archive is 78.7 GB. Nine notebooks would each extract essentially the same
~113 GB and each pay the same slow Drive read. Unpacking once and training nine
times amortizes that cost instead.

**Local disk usage:** about **113 GB** at peak — the train/validation
recordings, read straight off the Drive mount with no intermediate local copy
of the archive, and with the unused test split skipped. Step 5 explains both
choices.

## 1. Check the runtime

In [ ]:
!nvidia-smi

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No CUDA device visible. In the Colab VSCode extension, switch the connected runtime to a GPU "
    "(Runtime type > T4 GPU or better) and re-run this cell."
)
print(f"CUDA available: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Configuration

Edit these to match your setup. `DRIVE_PACKAGES_DIR` must point at the Drive
folder that *directly* contains `shared_standardized_recordings.zip` and the
`w*_h*.zip` files. Uploading a folder to Drive often nests it one level deeper
than you expect — step 5a will tell you if that happened.

In [ ]:
from pathlib import Path

# --- Git ---------------------------------------------------------------
GITHUB_REPO_URL = "https://github.com/amiradmehr/URV-Seizure-Pred.git"
GIT_BRANCH = "ethan-branch"
REPO_DIR = Path("/content/URV-Seizure-Pred")

# --- Packaged preprocessed data (see run_window_horizon_sweep.py --package) --
SWEEP_NAME = "windows_30-15-10_horizons_2-5-10"
DRIVE_PACKAGES_DIR = Path(
    f"/content/drive/MyDrive/seizeIT2_window_sweep_data/colab_packages/{SWEEP_NAME}"
)

# --- Sweep grid (must match the combinations that were packaged) -------
WINDOWS = [30, 15, 10]   # minutes
HORIZONS = [2, 5, 10]    # minutes

# Extra arguments forwarded verbatim to every train_eegnet_baseline.py call.
# Set ["--epochs", "1"] for a quick timing run before committing to the full
# sweep. Leave empty for the defaults (20 epochs, early stopping patience 6).
EXTRA_TRAIN_ARGS: list[str] = []

# Where trained outputs are copied back to Drive after each combo finishes,
# so a Colab disconnect never loses a completed combo's results.
DRIVE_RESULTS_DIR = Path(
    f"/content/drive/MyDrive/seizeIT2_window_sweep_data/sweep_results/{SWEEP_NAME}"
)
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 4. Clone the repository and install dependencies

In [ ]:
import subprocess

if REPO_DIR.exists():
    print(f"{REPO_DIR} already exists; pulling latest {GIT_BRANCH} instead of re-cloning.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", GIT_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", GIT_BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "reset", "--hard", f"origin/{GIT_BRANCH}"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", GIT_BRANCH, "--single-branch",
            GITHUB_REPO_URL, str(REPO_DIR),
        ],
        check=True,
    )

In [ ]:
# Colab already ships a CUDA-enabled torch build; installing this project's
# other dependencies without pulling in a new torch avoids Colab's GPU torch
# being silently swapped for a CPU-only or mismatched-CUDA wheel.
!pip install -q mne mne-bids braindecode scikit-learn tqdm
!pip install -q --no-deps -e "{REPO_DIR}"

In [ ]:
import sys

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import torch  # noqa: E402  (re-import to confirm CUDA survived dependency installs)
from seizure_prediction.seizeit2.config import build_config  # noqa: E402

assert torch.cuda.is_available(), "torch lost CUDA support after installing dependencies."
print("Imports OK. CUDA still available:", torch.cuda.get_device_name(0))

## 5. Unpack the preprocessed data

Every path recorded in a combo's `processed_shard_manifest.csv` is relative to
the repository root, and each zip was built the same way — so extracting
straight into `REPO_DIR` reproduces the exact `data/seizeit2/...` layout every
combo expects, with no path editing.

The three steps run in order for a reason:

* **5a** verifies the upload in seconds — wrong Drive path, missing file, or a
  truncated upload all surface here instead of an hour into 5c.
* **5b** unpacks the nine small combo packages. Their manifests are what decide
  which recordings 5c actually has to extract.
* **5c** unpacks the standardized recordings. This is the slow step.

Two things keep 5c inside the runtime's disk budget:

* **No archive is copied to local disk.** They are read in place off the Drive
  mount. A local copy would have to coexist with everything extracted out of
  it — 78.7 GB + 127.7 GB ≈ 206 GB, more than the runtime has, which is exactly
  how an earlier version of this notebook died with *"[Errno 28] No space left
  on device"*. The same bytes cross the Drive mount either way, so the copy
  bought nothing.
* **The test split is skipped.** `train_eegnet_baseline.py` reads only the
  `train` and `validation` splits — it ends by printing *"The held-out test
  split was not used."* — so 605 recordings and 15.3 GB never need to land on
  disk.

Peak usage lands near **113 GB**, leaving comfortable headroom for checkpoints.

**5c is resumable.** Any recording already on disk at its full expected size is
skipped, so if the Drive mount drops partway through (`Transport endpoint is
not connected`), just re-run the cell and it picks up where it stopped.

In [ ]:
import concurrent.futures
import shutil
import threading
import time
import zipfile

import pandas as pd


def disk_free_gb(path: Path = Path("/content")) -> float:
    return shutil.disk_usage(path).free / 1e9


def extract_zip(zip_path: Path, destination: Path) -> None:
    """Extract all of `zip_path` into `destination`, reading the zip in place.

    Used for the small per-combo packages (tens of MB), where the FUSE-mount
    read latency per member is negligible next to the total extraction time.
    """
    print(f"Extracting {zip_path.name} ({zip_path.stat().st_size / 1e6:.0f} MB) -> {destination}")
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(destination)


_thread_local_zip = threading.local()


def _thread_zip_handle(zip_path: Path) -> zipfile.ZipFile:
    """Return a ZipFile handle private to the calling thread.

    zipfile.ZipFile isn't documented as safe to share across threads for
    concurrent reads, so each worker thread gets and reuses its own handle
    instead of every extraction reopening (and re-reading the central
    directory of) the archive from scratch.
    """
    handle = getattr(_thread_local_zip, "handle", None)
    if handle is None or _thread_local_zip.path != zip_path:
        handle = zipfile.ZipFile(zip_path)
        _thread_local_zip.handle = handle
        _thread_local_zip.path = zip_path
    return handle


def index_archives(zip_paths: list[Path]) -> dict[str, tuple[Path, int]]:
    """Map every member name across `zip_paths` to (archive, uncompressed size).

    Later archives win on a duplicate name, so a supplement can override the
    main archive. Archives that do not exist are skipped rather than raising --
    the caller decides whether an absent one actually matters.
    """
    index: dict[str, tuple[Path, int]] = {}
    for zip_path in zip_paths:
        if not zip_path.exists():
            continue
        with zipfile.ZipFile(zip_path) as archive:
            for info in archive.infolist():
                index[info.filename] = (zip_path, info.file_size)
    return index


def extract_recordings(
    index: dict[str, tuple[Path, int]],
    destination: Path,
    names: list[str],
    *,
    max_workers: int = 8,
    progress_every: int = 100,
) -> None:
    """Extract `names` into `destination`, reading each archive in place.

    Deliberately does NOT copy an archive to local disk first: a local copy has
    to coexist with everything extracted out of it, which needs far more free
    disk than a Colab runtime has. See the section notes above.

    Members already present at their full expected size are skipped, so
    re-running after a dropped Drive mount resumes instead of restarting.
    """
    pending: list[str] = []
    for name in names:
        _, expected_size = index[name]
        target = destination / name
        if target.exists() and target.stat().st_size == expected_size:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        pending.append(name)

    already_done = len(names) - len(pending)
    if already_done:
        print(f"    {already_done} recording(s) already extracted; resuming with the rest.")
    if not pending:
        print("    Nothing left to extract.")
        return

    pending_gb = sum(index[name][1] for name in pending) / 1e9
    print(f"    Extracting {len(pending)} recording(s), {pending_gb:.1f} GB, {max_workers} workers...")

    completed = 0
    progress_lock = threading.Lock()
    start_time = time.monotonic()

    def _extract_one(name: str) -> None:
        nonlocal completed
        zip_path, _ = index[name]
        _thread_zip_handle(zip_path).extract(name, destination)
        with progress_lock:
            completed += 1
            if completed % progress_every == 0 or completed == len(pending):
                elapsed = time.monotonic() - start_time
                rate = completed / max(elapsed, 1e-9)
                remaining = (len(pending) - completed) / max(rate, 1e-9)
                print(
                    f"    {completed}/{len(pending)} extracted "
                    f"({elapsed / 60:.1f} min elapsed, ~{remaining / 60:.1f} min left, "
                    f"{disk_free_gb():.0f} GB free)",
                    flush=True,
                )

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        list(executor.map(_extract_one, pending))


def required_recording_members() -> list[str]:
    """Every standardized-recording path the packaged combos actually read.

    train_eegnet_baseline.py loads only the `train` and `validation` splits --
    it ends by printing "The held-out test split was not used." -- so the test
    split's recordings never need to reach local disk at all.

    The nine combos do NOT all reference the same recordings: a recording is
    only usable if it is long enough to yield at least one decision, so a
    shorter window admits recordings a longer one had to skip (w10 references
    2782 recordings where w30 references 2689). Hence the union across every
    combo rather than any single combo's manifest.

    Paths in processed_shard_manifest.csv use the packaging machine's separator
    (backslashes, if it was packaged on Windows), so they are normalised to the
    forward slashes zip members always use.
    """
    needed: set[str] = set()
    for window_minutes in WINDOWS:
        for horizon_minutes in HORIZONS:
            manifest_path = (
                build_config(window_minutes, horizon_minutes).manifests_dir
                / "processed_shard_manifest.csv"
            )
            manifest = pd.read_csv(manifest_path)
            used = manifest[manifest["split"].isin(["train", "validation"])]
            needed.update(used["X_path"].str.replace("\\", "/", regex=False))
    return sorted(needed)


print(f"Local disk free: {disk_free_gb():.1f} GB")

In [ ]:
# Step 5a: verify the upload before doing anything slow.
#
# Opening a ZipFile reads the archive's central directory from its tail, so a
# truncated or still-uploading file fails here in seconds instead of an hour
# into step 5c.
if not DRIVE_PACKAGES_DIR.exists():
    raise FileNotFoundError(
        f"{DRIVE_PACKAGES_DIR} does not exist.\n"
        "Find where the folder actually landed with:\n"
        "    !find /content/drive -name 'shared_standardized_recordings.zip'\n"
        "then correct DRIVE_PACKAGES_DIR in the Configuration cell. Uploading a "
        "folder to Drive commonly nests it one level deeper than expected."
    )

REQUIRED_ARCHIVES = [f"w{w}_h{h}.zip" for w in WINDOWS for h in HORIZONS]
REQUIRED_ARCHIVES.append("shared_standardized_recordings.zip")

# The main archive is written after the FIRST combination, but the shared cache
# keeps growing afterwards, so a sweep packaged before that was fixed is short
# the recordings only the shorter windows use. This supplement carries them.
# A sweep packaged with the fix needs no supplement, so it is optional here --
# step 5c reports definitively whether anything is actually missing.
OPTIONAL_ARCHIVES = ["shared_standardized_recordings_supplement.zip"]

problems: list[str] = []
for name in REQUIRED_ARCHIVES + OPTIONAL_ARCHIVES:
    path = DRIVE_PACKAGES_DIR / name
    if not path.exists():
        if name in REQUIRED_ARCHIVES:
            problems.append(f"{name}: missing from {DRIVE_PACKAGES_DIR}")
        else:
            print(f"  {name:52s} (absent -- optional)")
        continue
    try:
        with zipfile.ZipFile(path) as archive:
            member_count = len(archive.infolist())
    except zipfile.BadZipFile as error:
        problems.append(f"{name}: unreadable ({error}) -- re-upload it")
        continue
    print(f"  {name:52s} {path.stat().st_size / 1e9:7.2f} GB  {member_count:5d} members")

if problems:
    raise RuntimeError(
        "Problems with the uploaded archives:\n  " + "\n  ".join(problems)
    )

print(f"\nAll required archives present and readable. Disk free: {disk_free_gb():.1f} GB")

In [ ]:
# Step 5b: the nine small per-combo packages (labels, manifests, scaler).
# These come first because the next cell reads their manifests to decide which
# recordings it actually has to pull out of the big shared archive.
for window_minutes in WINDOWS:
    for horizon_minutes in HORIZONS:
        tag = f"w{window_minutes}_h{horizon_minutes}"
        manifest_path = (
            build_config(window_minutes, horizon_minutes).manifests_dir
            / "processed_shard_manifest.csv"
        )
        if manifest_path.exists():
            print(f"{tag}: already unpacked; skipping.")
            continue
        extract_zip(DRIVE_PACKAGES_DIR / f"{tag}.zip", REPO_DIR)

print(f"\nDisk free: {disk_free_gb():.1f} GB")

In [ ]:
# Step 5c: the standardized recordings. This is the slow step -- roughly 70 GB
# has to come across the Drive mount. Re-run it as-is if the mount drops; it
# resumes from whatever already made it to disk.
shared_zips = [
    DRIVE_PACKAGES_DIR / "shared_standardized_recordings.zip",
    DRIVE_PACKAGES_DIR / "shared_standardized_recordings_supplement.zip",
]

needed_members = required_recording_members()
archive_index = index_archives(shared_zips)

unavailable = [name for name in needed_members if name not in archive_index]
if unavailable:
    raise KeyError(
        f"{len(unavailable)} recording(s) referenced by a combo manifest are in "
        f"none of the uploaded archives, e.g. {unavailable[:3]}.\n"
        f"If {shared_zips[1].name} is not in Drive, upload it -- it holds the "
        "recordings the shorter-window combos need that the main archive predates."
    )

needed_gb = sum(archive_index[name][1] for name in needed_members) / 1e9
total_gb = sum(size for _, size in archive_index.values()) / 1e9
free_gb = disk_free_gb()

print(f"{len(needed_members)} recordings are used by the train/validation splits.")
print(
    f"  Extracting: {needed_gb:.1f} GB  "
    f"(skipping {total_gb - needed_gb:.1f} GB of test-split data)"
)
print(f"  Disk free:  {free_gb:.1f} GB")

if free_gb < needed_gb * 1.1:
    raise RuntimeError(
        f"Only {free_gb:.1f} GB free, but extracting the train/validation "
        f"recordings needs {needed_gb:.1f} GB plus headroom for checkpoints. "
        "Switch to a larger-disk runtime, or free space under /content first."
    )

start_time = time.monotonic()
extract_recordings(archive_index, REPO_DIR, needed_members)
print(f"\nDone in {(time.monotonic() - start_time) / 60:.1f} min.")
print(f"Disk free after extraction: {disk_free_gb():.1f} GB")

## 6. Train every combination

Runs `scripts/seizeit2/train_eegnet_baseline.py --device cuda` once per
(window, horizon) combination — the same training stage
`run_window_horizon_sweep.py` runs, just without the preprocessing stages that
already happened. A combo whose `metrics.json` already exists on Drive is
skipped, so a rerun after a Colab disconnect only trains what's left.

Each combo's results are copied to Drive the moment it finishes, so a
disconnect never costs you more than the combo in flight.

In [ ]:
import json

TRAIN_SCRIPT = REPO_DIR / "scripts" / "seizeit2" / "train_eegnet_baseline.py"
sweep_dir = REPO_DIR / "outputs" / "sweeps" / SWEEP_NAME
log_dir = sweep_dir / "logs"
log_dir.mkdir(parents=True, exist_ok=True)


def run_training(window_minutes: float, horizon_minutes: float) -> dict:
    tag = f"w{window_minutes}_h{horizon_minutes}"
    output_dir = sweep_dir / "models" / tag
    drive_metrics_path = DRIVE_RESULTS_DIR / tag / "metrics.json"

    if drive_metrics_path.exists():
        print(f"=== {tag}: already trained (found on Drive); skipping. ===")
        return json.loads(drive_metrics_path.read_text(encoding="utf-8"))

    print(f"\n=== {tag}: window={window_minutes}min, horizon={horizon_minutes}min ===")
    command = [
        sys.executable, "-u", str(TRAIN_SCRIPT),
        "--window-minutes", str(window_minutes),
        "--horizon-minutes", str(horizon_minutes),
        "--device", "cuda",
        "--output-dir", str(output_dir),
        *EXTRA_TRAIN_ARGS,
    ]
    print(f"    $ {' '.join(command)}")

    log_path = log_dir / f"{tag}_train.log"
    start_time = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for line in process.stdout:
            print(f"    | {line}", end="")
            log_file.write(line)
        process.wait()

    if process.returncode != 0:
        print(f"    FAILED: train_eegnet_baseline.py exited {process.returncode}")
        return {
            "window_minutes": window_minutes,
            "horizon_minutes": horizon_minutes,
            "experiment_tag": tag,
            "best_validation_average_precision": None,
            "best_epoch": None,
            "status": f"failed: exit code {process.returncode}",
            "wall_clock_seconds": None,
        }

    metrics = json.loads((output_dir / "metrics.json").read_text(encoding="utf-8"))
    result = {
        "window_minutes": window_minutes,
        "horizon_minutes": horizon_minutes,
        "experiment_tag": tag,
        "best_epoch": metrics["best_epoch"],
        # Every validation metric from the selected epoch, not just AP, so the
        # comparison below can show more than one dimension of quality.
        **{
            name: value
            for name, value in metrics.items()
            if name.startswith("best_validation_")
        },
        "status": "ok",
        "wall_clock_seconds": round(time.monotonic() - start_time, 1),
    }

    # Copy this combo's results to Drive immediately, so a disconnect partway
    # through the sweep never loses a combo that already finished.
    drive_combo_dir = DRIVE_RESULTS_DIR / tag
    shutil.copytree(output_dir, drive_combo_dir, dirs_exist_ok=True)
    print(f"    Backed up to Drive: {drive_combo_dir}")

    return result

In [ ]:
results = []
for window_minutes in WINDOWS:
    for horizon_minutes in HORIZONS:
        results.append(run_training(window_minutes, horizon_minutes))

results_path = sweep_dir / "sweep_results.csv"
results_frame = pd.DataFrame(results).sort_values(
    "best_validation_average_precision", ascending=False, na_position="last"
)
results_frame.to_csv(results_path, index=False)
shutil.copy(results_path, DRIVE_RESULTS_DIR / "sweep_results.csv")
print(f"\nSweep results: {results_path}")
results_frame

## 7. Evaluate on the held-out test split

Everything above reports **validation** AP. That is the metric training selects
the best epoch on and the metric the sweep ranks combos by — which is exactly
why it is optimistically biased: it has been looked at repeatedly, so the best
combo's validation AP is partly a measure of how well it fit the validation
split. `train_eegnet_baseline.py` never touches the test split, and says so on
its last line.

This step measures each trained model once on that held-out test split, using
the same probability calibration training applied, so test AP is directly
comparable to `best_validation_average_precision`.

**Disk:** step 5c deliberately skipped the test recordings (15.3 GB) because
they would not fit alongside the training data. Rather than extracting them all
now, `test_eegnet_baseline.py` streams them out of the Drive archives a batch of
recordings at a time and deletes each batch after scoring it. At 40 recordings
per batch, peak extra disk is about **2.5 GB** — comfortable even with ~18 GB
free. Predictions are pooled across batches and scored once at the end, so
batching does not change any number.

In [ ]:
TEST_SCRIPT = REPO_DIR / "scripts" / "seizeit2" / "test_eegnet_baseline.py"

# The test recordings were never extracted, so stream them from the archives.
SHARED_ARCHIVE_ARGS: list[str] = []
for archive_name in (
    "shared_standardized_recordings.zip",
    "shared_standardized_recordings_supplement.zip",
):
    archive_path = DRIVE_PACKAGES_DIR / archive_name
    if archive_path.exists():
        SHARED_ARCHIVE_ARGS += ["--shared-archive", str(archive_path)]

# 40 recordings per batch peaks near 2.5 GB of extra disk. Raise it if you have
# room to spare (fewer archive seeks), lower it if disk is tighter than that.
RECORDINGS_PER_BATCH = 40


def run_testing(window_minutes: float, horizon_minutes: float) -> dict:
    tag = f"w{window_minutes}_h{horizon_minutes}"
    output_dir = sweep_dir / "models" / tag
    drive_combo_dir = DRIVE_RESULTS_DIR / tag

    drive_test_metrics = drive_combo_dir / "test_metrics.json"
    if drive_test_metrics.exists():
        print(f"=== {tag}: already tested (found on Drive); skipping. ===")
        return json.loads(drive_test_metrics.read_text(encoding="utf-8"))

    # Prefer this session's checkpoint, but fall back to the copy backed up to
    # Drive so a combo trained in an earlier session can still be tested.
    checkpoint_path = output_dir / "best_model.pt"
    if not checkpoint_path.exists():
        checkpoint_path = drive_combo_dir / "best_model.pt"
    if not checkpoint_path.exists():
        print(f"=== {tag}: no checkpoint found; train it first. ===")
        return {"experiment_tag": tag, "status": "no checkpoint"}

    print(f"\n=== {tag}: testing {checkpoint_path} ===")
    command = [
        sys.executable, "-u", str(TEST_SCRIPT),
        "--window-minutes", str(window_minutes),
        "--horizon-minutes", str(horizon_minutes),
        "--checkpoint", str(checkpoint_path),
        "--output-dir", str(output_dir),
        "--device", "cuda",
        "--recordings-per-batch", str(RECORDINGS_PER_BATCH),
        *SHARED_ARCHIVE_ARGS,
    ]

    log_path = log_dir / f"{tag}_test.log"
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(
            command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )
        for line in process.stdout:
            print(f"    | {line}", end="")
            log_file.write(line)
        process.wait()

    if process.returncode != 0:
        print(f"    FAILED: test_eegnet_baseline.py exited {process.returncode}")
        return {"experiment_tag": tag, "status": f"failed: exit {process.returncode}"}

    record = json.loads((output_dir / "test_metrics.json").read_text(encoding="utf-8"))
    record["status"] = "ok"

    drive_combo_dir.mkdir(parents=True, exist_ok=True)
    for artifact in ("test_metrics.json", "test_summary.png"):
        if (output_dir / artifact).exists():
            shutil.copy(output_dir / artifact, drive_combo_dir / artifact)
    print(f"    Backed up test results to Drive: {drive_combo_dir}")

    return record

In [ ]:
test_results = []
for window_minutes in WINDOWS:
    for horizon_minutes in HORIZONS:
        test_results.append(run_testing(window_minutes, horizon_minutes))

test_frame = pd.DataFrame(test_results)
test_path = sweep_dir / "sweep_test_results.csv"
test_frame.to_csv(test_path, index=False)
shutil.copy(test_path, DRIVE_RESULTS_DIR / "sweep_test_results.csv")
print(f"\nTest results: {test_path}")

display_columns = [
    column
    for column in (
        "experiment_tag",
        "best_validation_average_precision",
        "test_average_precision",
        "test_average_precision_lift_over_prevalence",
        "test_roc_auc",
        "test_prevalence",
        "test_best_f1",
        "status",
    )
    if column in test_frame.columns
]
test_frame[display_columns].sort_values(
    "test_average_precision", ascending=False, na_position="last"
) if "test_average_precision" in test_frame.columns else test_frame

## 8. Compare combinations

Three views of the same nine runs:

1. **The selection gap** — validation AP (what training optimized, so
   optimistically biased) against test AP (measured once on held-out data).
   A combo whose test bar falls well short of its validation bar was flattered
   by the selection process.
2. **Multiple test metrics side by side** — no single number captures quality
   on a 0.1%-prevalence problem. AP and ROC AUC can disagree sharply, and the
   recall-at-capped-FPR bars answer the practical question directly.
3. **The full table**, sorted by test AP.

Read AP against **prevalence**, not against 1.0: at ~0.1% positives a useless
model scores about 0.001, so the `lift_over_prevalence` column is often the
clearest summary. ROC AUC looks generous under this much imbalance — it is
reported for comparability, not for ranking.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

comparison = results_frame.copy()
if "test_average_precision" in test_frame.columns:
    test_columns = [
        column
        for column in test_frame.columns
        if column.startswith("test_") or column == "experiment_tag"
    ]
    comparison = comparison.merge(
        test_frame[test_columns], on="experiment_tag", how="left"
    )
comparison = comparison.dropna(subset=["best_validation_average_precision"])
comparison = comparison.sort_values(
    "test_average_precision"
    if "test_average_precision" in comparison.columns
    else "best_validation_average_precision",
    ascending=False,
    na_position="last",
)

positions = np.arange(len(comparison))
has_test = "test_average_precision" in comparison.columns

figure, (gap_axis, metric_axis) = plt.subplots(1, 2, figsize=(17, 5.5))

# --- 1. validation vs test AP -------------------------------------------
width = 0.38 if has_test else 0.6
gap_axis.bar(
    positions - (width / 2 if has_test else 0),
    comparison["best_validation_average_precision"],
    width,
    label="Validation AP (selection metric)",
    color="#a6bddb",
)
if has_test:
    gap_axis.bar(
        positions + width / 2,
        comparison["test_average_precision"],
        width,
        label="Test AP (held out)",
        color="#2c7fb8",
    )
    prevalence = comparison["test_prevalence"].dropna()
    if len(prevalence):
        gap_axis.axhline(
            prevalence.mean(),
            color="#d95f02",
            linestyle="--",
            label=f"Prevalence baseline ({prevalence.mean():.4f})",
        )
gap_axis.set_xticks(positions)
gap_axis.set_xticklabels(comparison["experiment_tag"], rotation=45, ha="right")
gap_axis.set(
    title="Selection gap: validation vs held-out test AP",
    xlabel="Combination (window_horizon, minutes)",
    ylabel="Average precision",
)
gap_axis.legend(fontsize=9)
gap_axis.grid(axis="y", alpha=0.3)

# --- 2. several test metrics side by side --------------------------------
metric_columns = [
    ("test_average_precision", "Average precision", "#2c7fb8"),
    ("test_roc_auc", "ROC AUC", "#7fcdbb"),
    ("test_best_f1", "Best F1", "#c7e9b4"),
    ("test_recall_at_10pct_false_positive_rate", "Recall @ 10% FPR", "#fdae61"),
]
available = [entry for entry in metric_columns if entry[0] in comparison.columns]
if available:
    group_width = 0.8 / len(available)
    for offset, (column, label, color) in enumerate(available):
        metric_axis.bar(
            positions - 0.4 + group_width * (offset + 0.5),
            comparison[column],
            group_width,
            label=label,
            color=color,
        )
    metric_axis.set_xticks(positions)
    metric_axis.set_xticklabels(comparison["experiment_tag"], rotation=45, ha="right")
    metric_axis.set(
        title="Held-out test metrics by combination",
        xlabel="Combination (window_horizon, minutes)",
        ylabel="Metric value",
        ylim=(0.0, 1.0),
    )
    metric_axis.legend(fontsize=9)
    metric_axis.grid(axis="y", alpha=0.3)
else:
    metric_axis.text(
        0.5, 0.5, "Run step 7 to populate test metrics",
        ha="center", va="center", transform=metric_axis.transAxes,
    )
    metric_axis.axis("off")

figure.tight_layout()
plt.show()

In [ ]:
# The full comparison table, sorted by the honest (test) metric where available.
table_columns = [
    column
    for column in (
        "experiment_tag",
        "best_epoch",
        "best_validation_average_precision",
        "test_average_precision",
        "test_average_precision_lift_over_prevalence",
        "test_roc_auc",
        "test_brier_score",
        "test_best_f1",
        "test_precision_at_best_f1",
        "test_recall_at_best_f1",
        "test_specificity_at_best_f1",
        "test_recall_at_5pct_false_positive_rate",
        "test_recall_at_10pct_false_positive_rate",
        "test_prevalence",
        "wall_clock_seconds",
    )
    if column in comparison.columns
]
comparison_table = comparison[table_columns]
comparison_table.to_csv(sweep_dir / "sweep_comparison.csv", index=False)
shutil.copy(sweep_dir / "sweep_comparison.csv", DRIVE_RESULTS_DIR / "sweep_comparison.csv")
print(f"Comparison table: {sweep_dir / 'sweep_comparison.csv'}")
comparison_table.style.format(precision=4) if hasattr(comparison_table, "style") else comparison_table

## 9. (Optional) Persist everything back to Drive

Per-combo results are already copied to Drive as each combo finishes (steps 6
and 7).
This cell additionally archives the full sweep directory — including logs and
the comparison CSV — as one zip, useful for downloading a single file instead
of browsing the Drive folder.

In [ ]:
archive_path = shutil.make_archive(
    str(DRIVE_RESULTS_DIR / f"{SWEEP_NAME}_full"), "zip", root_dir=sweep_dir,
)
print(f"Archived full sweep directory to {archive_path}")